# Step 9 — Analisis subgroup (Tahap 18): SINTA, kontribusi jurnal, dan tren per kelompok SINTA

**Input:** `df_lda_topics_v2.csv` (step5 v2) dan `df_bertopic_topics_v2.csv` (step6 v2).

**Keputusan desain dan keterbatasan (dinyatakan sebelum hasil):**
1. **SINTA lengkap** (tidak ada missing; tiap jurnal punya satu nilai), sehingga analisis SINTA dilakukan. Namun SINTA adalah **peringkat jurnal saat ini**, bukan peringkat saat artikel terbit, dan **S1 hanya terdiri dari 6 jurnal** (mencakup jurnal internasional multidisiplin). Hasil per-SINTA adalah pola per-jurnal, bukan sifat "kualitas".
2. **Kelompok SINTA:** S1-S2 (20 jurnal), S3 (43), S4-S6 (220). S6 (594 artikel, 9 jurnal) terlalu kecil untuk berdiri sendiri; S1 dan S2 digabung agar cukup jurnal. Ubah di `SINTA_GROUPS` bila perlu.
3. **Artikel dalam satu jurnal tidak independen**, sehingga p-value chi-square terlalu optimistis. Yang dilaporkan: ukuran efek (Cramer's V) dan **interval kepercayaan bootstrap per jurnal** (jurnal yang di-resample, bukan artikel). Interval ini mengukur keragaman antar-jurnal.
4. Tren per kelompok memakai aturan yang sama dengan step8 (OLS pada proporsi 2015-2024). Ini analisis eksploratif, bukan uji hipotesis; tidak ada koreksi uji berganda.
5. Semua tabel dilaporkan untuk LDA (K=25) dan BERTopic reduced (25, hanya dokumen ber-topik).

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import jensenshannon
warnings.filterwarnings('ignore')

# ============ KONFIGURASI ============
SINTA_GROUPS = {'S1': 'S1-S2', 'S2': 'S1-S2', 'S3': 'S3', 'S4': 'S4-S6', 'S5': 'S4-S6', 'S6': 'S4-S6'}
GROUP_ORDER = ['S1-S2', 'S3', 'S4-S6']
LAST_FULL_YEAR = 2024
N_BOOT = 2000; SEED = 42
TOP_JOURNAL_FLAG = 25.0          # %: topik ditandai jika satu jurnal menyumbang >= ini
MIN_JOURNAL_DOCS = 100           # untuk daftar jurnal atipikal
F = dict(lda='df_lda_topics_v2.csv', bt='df_bertopic_topics_v2.csv')
# =====================================
missing = [v for v in F.values() if not os.path.exists(v)]; print('File belum ada:', missing); assert not missing
dl = pd.read_csv(F['lda']); db = pd.read_csv(F['bt'])
assert len(dl) == len(db) and (dl.title_raw.astype(str).values == db.title_raw.astype(str).values).all(), 'Korpus tidak identik'
d = pd.DataFrame({'tahun': dl.tahun, 'jurnal': dl.jurnal, 'sinta': dl.sinta, 'lda': dl.lda_topic_name, 'bt': db.bertopic_name, 'bt_id': db.bertopic_id_reduced})
assert d.sinta.notna().all() and set(d.sinta) <= set(SINTA_GROUPS), 'Ada nilai SINTA di luar peta kelompok'
assert (d.groupby('jurnal').sinta.nunique() == 1).all(), 'Ada jurnal dengan lebih dari satu nilai SINTA'
d['grp'] = d.sinta.map(SINTA_GROUPS)
dB = d[d.bt_id != -1].copy()                      # BERTopic: hanya dokumen ber-topik
print(f'Dokumen = {len(d)} | jurnal = {d.jurnal.nunique()} | BERTopic ber-topik = {len(dB)}')
json.dump(dict(sinta_groups=SINTA_GROUPS, n_boot=N_BOOT, seed=SEED, top_journal_flag_pct=TOP_JOURNAL_FLAG, last_full_year=LAST_FULL_YEAR), open('subgroup_config_v2.json', 'w'), indent=2)

## A. Gambaran SINTA dan Figure 12

In [ ]:
ov = d.groupby('sinta').agg(articles=('jurnal', 'size'), journals=('jurnal', 'nunique')); ov['share_pct'] = (100 * ov.articles / len(d)).round(1)
ov['articles_per_journal'] = (ov.articles / ov.journals).round(1)
ovg = d.groupby('grp').agg(articles=('jurnal', 'size'), journals=('jurnal', 'nunique')).loc[GROUP_ORDER]; ovg['share_pct'] = (100 * ovg.articles / len(d)).round(1)
print(ov.to_string()); print(); print(ovg.to_string()); ov.to_csv('sinta_overview_v2.csv')
by_year = pd.crosstab(d.tahun, d.sinta, normalize='index') * 100; by_year.round(1).to_csv('sinta_by_year_v2.csv'); print('\n', by_year.round(1).to_string())
print('\nS1: ', ', '.join(d[d.sinta == 'S1'].jurnal.value_counts().index.str[:60]))

fig, ax = plt.subplots(figsize=(8.5, 4.3)); bottom = np.zeros(len(by_year)); cols = plt.get_cmap('Blues')(np.linspace(0.9, 0.25, 6))
for c, s in zip(cols, sorted(by_year.columns)):
    ax.bar(by_year.index.astype(str), by_year[s], bottom=bottom, color=c, label=s, edgecolor='white', lw=.4); bottom += by_year[s].values
ax.set_ylabel('Share of articles (%)'); ax.set_xlabel('Year'); ax.legend(ncol=6, frameon=False, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.15))
ax.set_title('Figure 12. SINTA composition of the corpus by year (2025 partial)', fontsize=10); fig.tight_layout()
plt.savefig('figure12_sinta_by_year_v2.png', dpi=300, bbox_inches='tight'); plt.show()

## B. Topik x kelompok SINTA (Tahap 18B)
Untuk setiap topik: porsi artikel tiap kelompok yang jatuh ke topik itu, dengan interval bootstrap per jurnal (95%), dan selisih S1-S2 dikurangi S4-S6. Selisih dianggap **terlihat** hanya jika intervalnya tidak memuat nol. Cramer's V mengukur kuat-lemahnya asosiasi topik-SINTA secara keseluruhan.

In [ ]:
rng = np.random.default_rng(SEED)
def cramers_v(ct):
    chi2 = stats.chi2_contingency(ct)[0]; n = ct.values.sum(); return float(np.sqrt(chi2 / (n * (min(ct.shape) - 1))))

def topic_by_group(df, col):
    topics = sorted(df[col].unique())
    J = pd.crosstab(df.jurnal, df[col]).reindex(columns=topics, fill_value=0)
    jg = df.groupby('jurnal').grp.first(); boot = {}; est = {}
    for g in GROUP_ORDER:
        Jg = J.loc[jg[jg == g].index].values.astype(float); n = len(Jg)
        est[g] = Jg.sum(axis=0) / Jg.sum()
        idx = rng.integers(0, n, size=(N_BOOT, n)); s = Jg[idx].sum(axis=1); boot[g] = s / s.sum(axis=1, keepdims=True)
    diff = boot['S1-S2'] - boot['S4-S6']; rows = []
    for k, t in enumerate(topics):
        r = dict(Topic=t, corpus_share_pct=100 * J[t].sum() / J.values.sum())
        for g in GROUP_ORDER:
            r[f'{g}_pct'] = 100 * est[g][k]; r[f'{g}_ci_lo'] = 100 * np.percentile(boot[g][:, k], 2.5); r[f'{g}_ci_hi'] = 100 * np.percentile(boot[g][:, k], 97.5)
        r['diff_S1S2_minus_S4S6_pp'] = 100 * (est['S1-S2'][k] - est['S4-S6'][k])
        r['diff_ci_lo'] = 100 * np.percentile(diff[:, k], 2.5); r['diff_ci_hi'] = 100 * np.percentile(diff[:, k], 97.5)
        r['diff_visible'] = (r['diff_ci_lo'] > 0) or (r['diff_ci_hi'] < 0); rows.append(r)
    return pd.DataFrame(rows).sort_values('diff_S1S2_minus_S4S6_pp', ascending=False).reset_index(drop=True)

TL = topic_by_group(d, 'lda'); TB = topic_by_group(dB, 'bt')
TL.to_csv('topic_by_sinta_lda_v2.csv', index=False); TB.to_csv('topic_by_sinta_bertopic_v2.csv', index=False)
assoc = dict(lda_cramers_v=cramers_v(pd.crosstab(d.lda, d.grp)), bertopic_cramers_v=cramers_v(pd.crosstab(dB.bt, dB.grp)),
             lda_cramers_v_sinta6=cramers_v(pd.crosstab(d.lda, d.sinta)), bertopic_cramers_v_sinta6=cramers_v(pd.crosstab(dB.bt, dB.sinta)))
json.dump(assoc, open('sinta_association_v2.json', 'w'), indent=2); print(json.dumps(assoc, indent=2))
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 50)
show = ['Topic', 'corpus_share_pct', 'S1-S2_pct', 'S3_pct', 'S4-S6_pct', 'diff_S1S2_minus_S4S6_pp', 'diff_ci_lo', 'diff_ci_hi', 'diff_visible']
print('\nLDA:'); print(TL[show].round(2).to_string(index=False)); print('\nBERTopic:'); print(TB[show].round(2).to_string(index=False))
print(f"\nSelisih terlihat: LDA {int(TL.diff_visible.sum())}/{len(TL)} topik | BERTopic {int(TB.diff_visible.sum())}/{len(TB)} topik")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 7.2))
for ax, T, ttl in [(axes[0], TL, 'LDA (K=25)'), (axes[1], TB, 'BERTopic reduced (25)')]:
    M = np.log2(T[[f'{g}_pct' for g in GROUP_ORDER]].values / T[['corpus_share_pct']].values)
    im = ax.imshow(M, cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto')
    ax.set_xticks(range(3)); ax.set_xticklabels(GROUP_ORDER); ax.set_yticks(range(len(T))); ax.set_yticklabels([t[:46] for t in T.Topic], fontsize=7); ax.set_title(ttl, fontsize=10)
    for i in range(M.shape[0]):
        for j in range(3): ax.text(j, i, f'{T[f"{GROUP_ORDER[j]}_pct"].iloc[i]:.1f}', ha='center', va='center', fontsize=6)
fig.colorbar(im, ax=axes, shrink=.6, label='log2(group share / corpus share)  (red = over-represented)')
fig.suptitle('Figure 13. Topic representation by SINTA group (numbers = % of the group\'s articles)', fontsize=11)
plt.savefig('figure13_topic_by_sinta_v2.png', dpi=300, bbox_inches='tight'); plt.show()

## C. Kontribusi jurnal per topik (Tahap 18C)
`effective_journals` = 1 / sum(p^2) (jumlah jurnal 'setara'; makin kecil makin terkonsentrasi). Topik ditandai jika satu jurnal >= `TOP_JOURNAL_FLAG` %.

In [ ]:
def contribution(df, col):
    rows = []
    for t, g in df.groupby(col):
        p = g.jurnal.value_counts(normalize=True)
        rows.append(dict(Topic=t, n_docs=len(g), n_journals=g.jurnal.nunique(), top_journal=p.index[0], top_journal_sinta=g[g.jurnal == p.index[0]].sinta.iloc[0],
                         top_journal_share_pct=100 * p.iloc[0], top3_share_pct=100 * p.iloc[:3].sum(), effective_journals=1 / (p ** 2).sum(), flagged=100 * p.iloc[0] >= TOP_JOURNAL_FLAG))
    return pd.DataFrame(rows).sort_values('top_journal_share_pct', ascending=False).reset_index(drop=True)
CL, CB = contribution(d, 'lda'), contribution(dB, 'bt')
CL.to_csv('topic_journal_contribution_lda_v2.csv', index=False); CB.to_csv('topic_journal_contribution_bertopic_v2.csv', index=False)
pd.set_option('display.max_colwidth', 40)
print('LDA - 8 teratas:'); print(CL.head(8)[['Topic', 'n_docs', 'n_journals', 'top_journal', 'top_journal_share_pct', 'top3_share_pct', 'effective_journals', 'flagged']].round(1).to_string(index=False))
print('\nBERTopic - 8 teratas:'); print(CB.head(8)[['Topic', 'n_docs', 'n_journals', 'top_journal', 'top_journal_share_pct', 'top3_share_pct', 'effective_journals', 'flagged']].round(1).to_string(index=False))
print(f"\nTopik ditandai (>= {TOP_JOURNAL_FLAG}% satu jurnal): LDA {int(CL.flagged.sum())}, BERTopic {int(CB.flagged.sum())}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.8), sharex=True)
for ax, T, ttl in [(axes[0], CL, 'LDA (K=25)'), (axes[1], CB, 'BERTopic reduced (25)')]:
    T = T.sort_values('top_journal_share_pct'); ax.barh([t[:44] for t in T.Topic], T.top_journal_share_pct, color=np.where(T.flagged, '#C44E52', '#4C72B0'))
    ax.axvline(TOP_JOURNAL_FLAG, color='k', ls='--', lw=.8); ax.tick_params(axis='y', labelsize=7); ax.set_xlabel('Share of the topic from its largest journal (%)'); ax.set_title(ttl, fontsize=10)
fig.suptitle('Figure 14. Journal concentration per topic (red = flagged)', fontsize=11); fig.tight_layout(); plt.savefig('figure14_journal_concentration_v2.png', dpi=300, bbox_inches='tight'); plt.show()

## D. Profil topik jurnal (jurnal terbesar dan jurnal paling atipikal)
Jarak Jensen-Shannon antara profil topik suatu jurnal dan profil seluruh korpus. Jurnal dengan jarak besar memiliki isi yang jauh dari korpus dan patut diperiksa terkait cakupan (scope).

In [ ]:
topics_l = sorted(d.lda.unique()); Jl = pd.crosstab(d.jurnal, d.lda).reindex(columns=topics_l, fill_value=0)
corp = Jl.sum(axis=0).values / Jl.values.sum(); n_j = Jl.sum(axis=1)
js = pd.Series([jensenshannon(Jl.loc[j].values / n_j[j], corp, base=2) for j in Jl.index], index=Jl.index)
top1 = Jl.idxmax(axis=1); top1_share = Jl.max(axis=1) / n_j
topics_b = sorted(dB.bt.unique()); Jb = pd.crosstab(dB.jurnal, dB.bt).reindex(index=Jl.index, columns=topics_b, fill_value=0); nb_ = Jb.sum(axis=1).replace(0, np.nan)
prof = pd.DataFrame({'sinta': d.groupby('jurnal').sinta.first(), 'n_articles': n_j, 'lda_top_topic': top1, 'lda_top_share_pct': 100 * top1_share,
                     'bt_top_topic': Jb.idxmax(axis=1).where(nb_.notna()), 'bt_top_share_of_assigned_pct': 100 * Jb.max(axis=1) / nb_,
                     'bt_outlier_pct': 100 * (1 - nb_ / n_j), 'js_distance_to_corpus': js}).reset_index().rename(columns={'index': 'jurnal'})
prof.sort_values('n_articles', ascending=False).to_csv('journal_topic_profile_v2.csv', index=False)
pd.set_option('display.max_colwidth', 42)
cols = ['jurnal', 'sinta', 'n_articles', 'lda_top_topic', 'lda_top_share_pct', 'bt_top_topic', 'js_distance_to_corpus']
print('15 jurnal terbesar:'); print(prof.sort_values('n_articles', ascending=False).head(15)[cols].round(2).to_string(index=False))
print(f'\n10 jurnal paling atipikal (n >= {MIN_JOURNAL_DOCS}):'); print(prof[prof.n_articles >= MIN_JOURNAL_DOCS].sort_values('js_distance_to_corpus', ascending=False).head(10)[cols].round(2).to_string(index=False))
print('\nJarak JS median seluruh jurnal (n >= %d): %.2f' % (MIN_JOURNAL_DOCS, prof[prof.n_articles >= MIN_JOURNAL_DOCS].js_distance_to_corpus.median()))

## E. Tren per kelompok SINTA (Tahap 18A x 18B)
Apakah topik yang naik atau turun di seluruh korpus juga naik atau turun di setiap kelompok SINTA? `same_sign_all` = slope kelompok searah dengan slope keseluruhan di ketiga kelompok. **Standarisasi komposisi:** porsi topik per tahun dihitung ulang dengan bobot kelompok SINTA yang dibuat tetap (bauran 2015-2024 keseluruhan), sehingga pergeseran bauran SINTA tidak ikut terhitung. `composition_driven` = tanda slope berubah (hanya jika salah satu slope >= 0,15 pp/tahun, agar noise pada slope nyaris nol tidak ikut ditandai), atau > 50% slope agregat (|slope| >= 0,3 pp) berasal dari pergeseran bauran. Bauran S4-S6 naik dari ~41% (2015) menjadi ~68% (2024), sehingga topik yang kaya di S4-S6 tampak naik secara agregat walau tidak naik di dalam kelompok (paradoks Simpson). Tabel ini eksploratif.

In [ ]:
def slopes(df, col, label):
    ys = np.arange(2015, LAST_FULL_YEAR + 1); topics = sorted(df[col].unique()); P = {}; out = {}
    for name, sub in [('all', df)] + [(g, df[df.grp == g]) for g in GROUP_ORDER]:
        c = pd.crosstab(sub.tahun, sub[col]).reindex(index=ys, columns=topics, fill_value=0); P[name] = c.div(c.sum(axis=1), axis=0).fillna(0)
        out[name] = {t: stats.linregress(ys, P[name][t].values) for t in topics}
    # standarisasi langsung: bobot kelompok dibuat TETAP (komposisi SINTA 2015-2024 keseluruhan) agar pergeseran bauran SINTA tidak ikut terhitung
    w = np.array([((df.grp == g) & (df.tahun <= LAST_FULL_YEAR)).sum() for g in GROUP_ORDER], float); w /= w.sum()
    std = sum(wi * P[g] for wi, g in zip(w, GROUP_ORDER)); rows = []
    for t in topics:
        r = dict(Topic=t, slope_all_pp=100 * out['all'][t].slope, p_all=out['all'][t].pvalue)
        for g in GROUP_ORDER: r[f'slope_{g}_pp'] = 100 * out[g][t].slope; r[f'p_{g}'] = out[g][t].pvalue
        sg = np.sign([out[g][t].slope for g in GROUP_ORDER]); r['same_sign_all'] = bool((sg == np.sign(out['all'][t].slope)).all())
        r['n_groups_p<0.10_same_sign'] = int(sum((out[g][t].pvalue < 0.10) and (np.sign(out[g][t].slope) == np.sign(out['all'][t].slope)) for g in GROUP_ORDER))
        ss = stats.linregress(ys, std[t].values); r['slope_standardized_pp'] = 100 * ss.slope; r['p_standardized'] = ss.pvalue
        r['composition_effect_pp'] = r['slope_all_pp'] - r['slope_standardized_pp']
        r['composition_driven'] = bool(((np.sign(r['slope_all_pp']) != np.sign(r['slope_standardized_pp'])) and max(abs(r['slope_all_pp']), abs(r['slope_standardized_pp'])) >= 0.15) or
                                       ((abs(r['composition_effect_pp']) > 0.5 * abs(r['slope_all_pp'])) and abs(r['slope_all_pp']) >= 0.3)); rows.append(r)
    return pd.DataFrame(rows).sort_values('slope_all_pp', ascending=False).reset_index(drop=True)
SL, SB = slopes(d, 'lda', 'LDA'), slopes(dB, 'bt', 'BERTopic')
SL.to_csv('trend_by_sinta_lda_v2.csv', index=False); SB.to_csv('trend_by_sinta_bertopic_v2.csv', index=False)
cols = ['Topic', 'slope_all_pp', 'slope_S1-S2_pp', 'slope_S3_pp', 'slope_S4-S6_pp', 'slope_standardized_pp', 'composition_driven', 'same_sign_all', 'n_groups_p<0.10_same_sign']
print('LDA:'); print(SL[cols].round(2).to_string(index=False)); print('\nBERTopic:'); print(SB[cols].round(2).to_string(index=False))
print(f"\nSearah di ketiga kelompok: LDA {int(SL.same_sign_all.sum())}/{len(SL)} | BERTopic {int(SB.same_sign_all.sum())}/{len(SB)}")
big = lambda T: T[T.slope_all_pp.abs() >= 0.3]
print(f"Topik dengan tren yang dipengaruhi bauran SINTA (composition_driven): LDA {int(SL.composition_driven.sum())}, BERTopic {int(SB.composition_driven.sum())}")
print(f"Di antara topik dengan |slope| >= 0,3 pp/tahun: LDA {int(big(SL).same_sign_all.sum())}/{len(big(SL))} searah di ketiga kelompok | BERTopic {int(big(SB).same_sign_all.sum())}/{len(big(SB))}")

## F. File yang dihasilkan
`subgroup_config_v2.json`, `sinta_overview_v2.csv`, `sinta_by_year_v2.csv`, `sinta_association_v2.json`, `topic_by_sinta_lda_v2.csv`, `topic_by_sinta_bertopic_v2.csv`,
`topic_journal_contribution_lda_v2.csv`, `topic_journal_contribution_bertopic_v2.csv`, `journal_topic_profile_v2.csv`, `trend_by_sinta_lda_v2.csv`, `trend_by_sinta_bertopic_v2.csv`,
`figure12_sinta_by_year_v2.png`, `figure13_topic_by_sinta_v2.png`, `figure14_journal_concentration_v2.png`.